# NB-02_controles_y_ajustes_iniciales

Process Flow SAS: **Controles y ajustes iniciales** — `PFD-I32F27oZ8ICuby6y`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

ANIO = None  # &ANIO — nadie declaró su valor
TRIM = None  # &TRIM — nadie declaró su valor
anio = None  # &anio — nadie declaró su valor

faltantes = [n for n, v in {"ANIO": ANIO, "TRIM": TRIM, "anio": anio}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text
import bcchapi

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=GOBGENER;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-02_controles_y_ajustes_iniciales.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## S1

Arma los insumos de síntesis del trimestre: intereses SIFMI por sector, promedio de dividendos de hogares del periodo de coyuntura, agregados macro trimestrales descargados del Banco Central (PIB y nueve series de cuentas nacionales), utilidades reinvertidas, cuentas por cobrar y los ajustes varios, incluidos los bonos emitidos en el exterior por el sector financiero no regulado / Reclasifica los bonos de renta fija externa como préstamos de largo plazo con signo invertido y traslada el activo AF.32 del resto del mundo desde el agente 36 hacia empresas, acumulando ambos ajustes en la base de ajustes varios

*confianza: medium · verificador: approve · SAS: PROC IMPORT XLSX + PROC SQL (UNION ALL, GROUP BY, DELETE/UPDATE) + PROC HTTP a la API BDE + PROC DATASETS APPEND + DATA step SET + PROC SQL UPDATE + CREATE TABLE con GROUP BY + DATA step SET (append a tabla de BD)*

In [ ]:
# ========= S1 =========
# /*IMPORTA DATA*/
# proc import datafile=".../CONTROLES/SIFMI.xlsx" dbms=XLSX out=SIFMI sheet="SIFMI_SAS" getnames=YES
ruta_sifmi = Path("data") / "CONTROLES" / "SIFMI.xlsx"
sifmi = pd.read_excel(ruta_sifmi, sheet_name="SIFMI_SAS")
_log("sifmi", sifmi)


In [ ]:
# /*CREA BASE DE DATOS SIFMI. CIERRE 2021: INCORPORA SECTORES GOB, SEGUROS Y AUXILIARES*/
# Las 12 ramas del UNION ALL: (sector, c_cagente, c_entrada, columnas origen, signo)
_fecha_hoy = pd.Timestamp.today().normalize()
_ramas_sifmi = [
    (51, "321", "D", ["Empresas_pagados"], -1),
    (511, "321", "D", ["Hogares_pagados"], -1),
    (41, "321", "D", ["Gob_pagados"], -1),
    (35, "321", "D", ["Seg_pagados"], -1),
    (36, "321", "D", ["Aux_pagados"], -1),
    (321, "53", "H", ["Hogares_pagados", "Empresas_pagados", "Gob_pagados", "Seg_pagados", "Aux_pagados"], -1),
    (51, "321", "H", ["Empresas_recibidos"], 1),
    (511, "321", "H", ["Hogares_recibidos"], 1),
    (41, "321", "H", ["Gob_recibidos"], 1),
    (35, "321", "H", ["Seg_recibidos"], 1),
    (36, "321", "H", ["Aux_recibidos"], 1),
    (321, "53", "D", ["Hogares_recibidos", "Empresas_recibidos", "Gob_recibidos", "Seg_recibidos", "Aux_recibidos"], 1),
]
tablas_sifmi = pd.concat(
    [
        pd.DataFrame({
            "MONEDA": "P",
            "AÑO": sifmi["Año"],
            "TRIM": sifmi["Trimestre"],
            "SECTOR": _sector,
            "C_CUENTA": "YG",
            "C_CAGENTE": _cagente,
            "C_ENTRADA": _entrada,
            "DATO": sifmi[_cols].sum(axis=1) * _signo,
            "C_SCN": "D.41",
            "N_SCN": "Intereses",
            "FUENTE": "DI_Aj_SIFMI",
            "FECHA": _fecha_hoy,
        })
        for _sector, _cagente, _entrada, _cols, _signo in _ramas_sifmi
    ],
    ignore_index=True,
)
_log("tablas_sifmi", tablas_sifmi)


In [ ]:
# CREATE TABLE TABLAS.SIFMI AS ... -> la tabla la crea este flujo desde el Excel
# en cada corrida (created_tables): se reemplaza entera
_cols_sifmi = ["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CUENTA", "C_CAGENTE", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "FECHA"]
tablas_sifmi[_cols_sifmi].to_sql("SIFMI", engine, schema="dbo", if_exists="replace", index=False)
_log("CREATE TABLAS.dbo.SIFMI", tablas_sifmi)


In [ ]:
# PROC SQL; DELETE FROM TABLAS.SIFMI WHERE AÑO=.;  (faltante SAS = NULL en SQL Server)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.SIFMI WHERE AÑO IS NULL"))
    _log("DELETE TABLAS.dbo.SIFMI", res.rowcount)


In [ ]:
# /******************DIVIDENDOS HOGARES********************/
# /*CALCULA PROMEDIO DEL TRIMESTRE A TRABAJAR. AJUSTE DE INICIO DEL PERIODO EN EL
#   PROCESO DE SÍNTESIS PARA EL PERIODO DE COYUNTURA*/
work_conn.execute(text("DROP TABLE IF EXISTS #rp_hh_sum"))
sql_rp_hh_sum = f"""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_CAGENTE, T1.C_ENTRADA,
       T1.C_SCN, T1.N_SCN, T1.FUENTE, SUM(T1.DATO) AS DATO
INTO #rp_hh_sum
FROM TABLAS.dbo.RP_HH T1
WHERE T1.AÑO >= 2008 AND T1.TRIM = {int(TRIM)}
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_CAGENTE, T1.C_ENTRADA,
         T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_rp_hh_sum))


In [ ]:
# /*PARA ELIMINAR PERIODO DE COYUNTURA EN CASO QUE SE CORRA ESTE PROG VARIAS VECES*/
res = work_conn.execute(text(f"DELETE FROM #rp_hh_sum WHERE AÑO = {int(ANIO)} AND TRIM = {int(TRIM)}"))
_log("DELETE #rp_hh_sum coyuntura", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #rp_hh_av"))
sql_rp_hh_av = f"""
SELECT T1.MONEDA, {int(ANIO)} AS AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_CAGENTE, T1.C_ENTRADA,
       T1.C_SCN, T1.N_SCN, T1.FUENTE, AVG(T1.DATO) AS DATO,
       CAST(GETDATE() AS date) AS FECHA, 'P' AS PROC
INTO #rp_hh_av
FROM #rp_hh_sum T1
GROUP BY T1.MONEDA, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_CAGENTE, T1.C_ENTRADA,
         T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_rp_hh_av))
rp_hh_av = pd.read_sql(text("SELECT * FROM #rp_hh_av"), work_conn)
_log("rp_hh_av", rp_hh_av)


In [ ]:
# /*ELIMINA DATOS DE COYUNTURA EN TABLA PRINCIPAL*/
sql_del_rp_hh = """
DELETE t
FROM TABLAS.dbo.RP_HH t
WHERE EXISTS (
    SELECT 1 FROM #rp_hh_av a
    WHERE a.AÑO = t.AÑO AND a.TRIM = t.TRIM AND a.PROC = t.PROC
)
"""
res = work_conn.execute(text(sql_del_rp_hh))
_log("DELETE TABLAS.dbo.RP_HH coyuntura", res.rowcount)


In [ ]:
# /*ANEXA PROMEDIO DIVIDENDOS A BASE RP_HH*/
# proc datasets; append base=tablas.RP_HH data=WORK.RP_HH_AV force;
_cols_rp_hh = "MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN, FUENTE, DATO, FECHA, PROC"
sql_append_rp_hh = f"""
INSERT INTO TABLAS.dbo.RP_HH ({_cols_rp_hh})
SELECT {_cols_rp_hh}
FROM #rp_hh_av
"""
res = work_conn.execute(text(sql_append_rp_hh))
_log("APPEND TABLAS.dbo.RP_HH", res.rowcount)


In [ ]:
# /******************BASE CON VARIABLES DE LAS CNT A UTILIZAR PARA AJUSTAR DATOS DE LAS CNSI********************/
# /*IMPORTA DATA DESDE ARCHIVO GENERADO POR API*/ /*obtiene datos directo desde API web*/
# El host si3.bcentral.cl se consulta con el paquete oficial bcchapi (decisión del proyecto)
_bcch = bcchapi.Siete(os.environ["BDE_USER"], os.environ["BDE_PASS"])


def _serie_bde(codigo: str) -> pd.DataFrame:
    """Descarga una serie de la API BDE y replica el parseo del SAS:
    AÑO = SUBSTR(indexDateString,7,4), TRIM = SUBSTR(indexDateString,4,2), DATO = value*1000."""
    _df = _bcch.cuadro(series=[codigo], nombres=["value"]).reset_index()
    _df.columns = ["indexDateString", "value"]
    _fechas = pd.to_datetime(_df["indexDateString"], errors="coerce")
    _out = pd.DataFrame({
        "AÑO": _fechas.dt.year,
        "TRIM": _fechas.dt.month,
        "DATO": pd.to_numeric(_df["value"], errors="coerce") * 1000,
    })
    return _out[_out["AÑO"].notna()].reset_index(drop=True)


In [ ]:
# /*PIB a precios corrientes*/
pib = _serie_bde("F032.PIB.FLU.N.CLP.EP18.Z.Z.0.T")
pib["FECHA"] = pd.Timestamp.today().normalize()
pib = pib[["AÑO", "TRIM", "DATO", "FECHA"]]
# TABLAS.PIB está en created_tables: la crea este flujo en cada corrida
pib.to_sql("PIB", engine, schema="dbo", if_exists="replace", index=False)
_log("CREATE TABLAS.dbo.PIB", pib)


In [ ]:
# UPDATE tablas.PIB SET TRIM=2 WHERE TRIM=4;  ... TRIM=3 WHERE TRIM=7;  ... TRIM=4 WHERE TRIM=10;
with engine.begin() as conn:
    for _desde, _hasta in [(4, 2), (7, 3), (10, 4)]:
        conn.execute(text(f"UPDATE TABLAS.dbo.PIB SET TRIM = {int(_hasta)} WHERE TRIM = {int(_desde)}"))
_log("UPDATE TABLAS.dbo.PIB TRIM", 3)


In [ ]:
# Series CNT desde la API BDE: (nombre, código serie, sector, c_cuenta, c_entrada, c_scn, n_scn)
_series_cnt = [
    # /*Ingreso de los factores recibidos*/
    ("serie_1", "F033.IRM.FLU.N.CLP.EP18.0.T", 6, "YG", "D", "D.4", "Ingreso de factores recibidos del RM"),
    # /*Ingreso de los factores pagados al RM*/
    ("serie_2", "F033.IRMP.FLU.N.CLP.EP18.0.T", 6, "YG", "H", "D.4", "Ingreso de factores pagados al RM"),
    # /*Transferencias corrientes recibidas del exterior*/
    ("serie_3", "F033.TCE.FLU.N.CLP.EP18.0.T", 6, "YG", "D", "D.7", "Transferencias corrientes recibidos del RM"),
    # /*Transferencias corrientes pagadas al exterior*/
    ("serie_4", "F033.TCEP.FLU.N.CLP.EP18.0.T", 6, "YG", "H", "D.7", "Transferencias corrientes pagados al RM"),
    # /*Ahorro externo*/
    ("serie_5", "F033.AEX.FLU.N.CLP.EP18.0.T", 6, "YG", "D", "B.8", "Ahorro externo"),
    # /*Formacion_bruta_capital fijo*/
    ("serie_6", "F033.FKF.FLU.N.CLP.EP18.0.T", 53, "Capital", "D", "P.51", "Formación bruta de capital fijo"),
    # /*Variacion_Existencias*/
    ("serie_7", "F033.VAX.FLU.N.CLP.EP18.0.T", 53, "Capital", "D", "P.52", "Variación de existencias"),
    # /*Exportaciones*/
    ("serie_8", "F033.XBS.FLU.N.CLP.EP18.0.T", 6, "Producción", "D", "P.7", "Importación de bienes y servicios"),
    # /*Importaciones*/
    ("serie_9", "F033.IBS.FLU.N.CLP.EP18.0.T", 6, "Producción", "H", "P.6", "Exportación de bienes y servicios"),
]
_hoy_cnt = pd.Timestamp.today().normalize()
_series_dfs = {}
for _nombre, _codigo, _sector, _cuenta, _entrada, _cscn, _nscn in _series_cnt:
    _s = _serie_bde(_codigo)
    _s = _s[_s["AÑO"] >= 2003].copy()
    _s["SECTOR"] = _sector
    _s["C_CUENTA"] = _cuenta
    _s["C_ENTRADA"] = _entrada
    _s["C_SCN"] = _cscn
    _s["N_SCN"] = _nscn
    _s["FUENTE"] = "CNT"
    _s["FECHA"] = _hoy_cnt
    _series_dfs[_nombre] = _s[["AÑO", "TRIM", "SECTOR", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "FECHA"]]
serie_1, serie_2, serie_3 = _series_dfs["serie_1"], _series_dfs["serie_2"], _series_dfs["serie_3"]
serie_4, serie_5, serie_6 = _series_dfs["serie_4"], _series_dfs["serie_5"], _series_dfs["serie_6"]
serie_7, serie_8, serie_9 = _series_dfs["serie_7"], _series_dfs["serie_8"], _series_dfs["serie_9"]
_log("series CNT descargadas", len(_series_dfs))


In [ ]:
# /*une base de datos CNT*/
# data tablas.CNT; set serie_1 ... serie_9;
cnt = pd.concat([serie_1, serie_2, serie_3, serie_4, serie_5, serie_6, serie_7, serie_8, serie_9], ignore_index=True)
# TABLAS.CNT está en created_tables: la crea este flujo en cada corrida
cnt.to_sql("CNT", engine, schema="dbo", if_exists="replace", index=False)
_log("CREATE TABLAS.dbo.CNT", cnt)


In [ ]:
# UPDATE TABLAS.CNT SET TRIM=2 WHERE TRIM=4;  ... TRIM=3 WHERE TRIM=7;  ... TRIM=4 WHERE TRIM=10;
with engine.begin() as conn:
    for _desde, _hasta in [(4, 2), (7, 3), (10, 4)]:
        conn.execute(text(f"UPDATE TABLAS.dbo.CNT SET TRIM = {int(_hasta)} WHERE TRIM = {int(_desde)}"))
_log("UPDATE TABLAS.dbo.CNT TRIM", 3)


In [ ]:
# proc sql; drop table serie_1, ..., serie_9;
for _nombre in list(_series_dfs):
    del _series_dfs[_nombre]
del serie_1, serie_2, serie_3, serie_4, serie_5, serie_6, serie_7, serie_8, serie_9


In [ ]:
# /******************UTILIDADES REINVERTIDAS DEL SECTOR FINANCIERO********************/
# /*IMPORTA DATA FINAL DE UTILIDADES REINVERTIDAS PAGADAS POR EL SECTOR, NUEVO CALCULO
#   TRIMESTRAL CR18. ADEMÁS INCORPORA APERTURA ENTRE BANCOS Y SEGUROS*/
ruta_ur_sf = Path("data") / "INFO_AUX" / "UT_REINVERTIDAS_CR18.xlsx"
ur_sf_cr18 = pd.read_excel(ruta_ur_sf, sheet_name="UR_SF", skiprows=1)
# PROC IMPORT out=TABLAS.UR_SF_CR18 replace -> reemplaza el contenido de la tabla existente
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.UR_SF_CR18"))
    _log("DELETE TABLAS.dbo.UR_SF_CR18", res.rowcount)
ur_sf_cr18.to_sql("UR_SF_CR18", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# /*NO SE INCORPORA APERTURA PORQUE AFECTA MUCHO EL PTMO NETO DE LOS SEGUROS*/
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.UR_SF_CR18 SET SECTOR = 321 WHERE SECTOR = 35"))
    _log("UPDATE TABLAS.dbo.UR_SF_CR18 SECTOR", res.rowcount)


In [ ]:
# /*IMPORTA DATA DE CCAS*/
ruta_t_ccast = Path("data") / "INFO_AUX" / "T_CCAST.xlsx"
t_ccast = pd.read_excel(ruta_t_ccast, sheet_name="T_CCAST")
# data tablas.T_CCAST; set WORK.T_CCAST; -> TABLAS.T_CCAST la crea este flujo (created_tables)
t_ccast.to_sql("T_CCAST", engine, schema="dbo", if_exists="replace", index=False)
_log("CREATE TABLAS.dbo.T_CCAST", t_ccast)


In [ ]:
# /*ELIMINA DE AJUSTE BONOS AÑO DE COYUNTURA PARA RECALCULAR DENUEVO POR CAMBIO DE CUENTAS INDIVIDUALES*/
with engine.begin() as conn:
    res = conn.execute(text(f"DELETE FROM TABLAS.dbo.AJUSTE_BONOS WHERE AÑO >= {int(ANIO)}"))
    _log("DELETE TABLAS.dbo.AJUSTE_BONOS", res.rowcount)


In [ ]:
# /*IMPORTA AJUSTES VARIOS DEP Y ACCIONES*/
ruta_aj_cnsi = Path("data") / "INFO_AUX" / "aj_cnsi.xlsx"
aj_varios = pd.read_excel(ruta_aj_cnsi, sheet_name="AJUSTES_VARIOS")
# PROC IMPORT out=TABLAS.AJ_VARIOS replace -> reemplaza el contenido de la tabla existente
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.AJ_VARIOS"))
    _log("DELETE TABLAS.dbo.AJ_VARIOS", res.rowcount)
aj_varios.to_sql("AJ_VARIOS", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# /*CIERRE 2021: IMPORTA AJUSTES TRANSFERENCIAS CORREINTES DE EMPRESAS POR CDR18*/
aj_d443_cr18 = pd.read_excel(ruta_aj_cnsi, sheet_name="base_aj_d443_cr18")
# proc sql; delete from AJ_d443_CR18 where año=.;
aj_d443_cr18 = aj_d443_cr18[aj_d443_cr18["año"].notna()].reset_index(drop=True)
_log("aj_d443_cr18", aj_d443_cr18)


In [ ]:
# DATA TABLAS.AJ_VARIOS; SET TABLAS.AJ_VARIOS AJ_d443_CR18;
# El DATA step reescribe la tabla con lo que ya tenía + el nuevo bloque: en la base
# eso es un append del bloque nuevo sobre el contenido vigente
aj_d443_cr18.to_sql("AJ_VARIOS", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.AJ_VARIOS (aj_d443_cr18)", aj_d443_cr18)


In [ ]:
# PROC SQL; DROP TABLE AJ_d443_CR18;
del aj_d443_cr18


In [ ]:
# /*CIERRE 2021: INCORPORA AJUSTE A FBCF SECTOR FINANCIERO*/
ruta_fbcf_sf = Path("data") / "INFO_AUX" / "aj_fbcf_sf.xlsx"
fbcf_sf = pd.read_excel(ruta_fbcf_sf, sheet_name="BASE")
# PROC IMPORT out=tablas.FBCF_SF replace -> reemplaza el contenido de la tabla existente
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.FBCF_SF"))
    _log("DELETE TABLAS.dbo.FBCF_SF", res.rowcount)
fbcf_sf.to_sql("FBCF_SF", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# /******************LIMPIA TABLAS TEMPORALES DEL FLUJO DE PROCESO********************/
# PROC SQL; DROP TABLE SIFMI, RP_HH_SUM, RP_HH_AV, ... , T_CCAST, ... , DEP_HH_FM;
for _tmp in ["#rp_hh_sum", "#rp_hh_av"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_tmp}"))
for _obj in ["sifmi", "rp_hh_av", "t_ccast"]:
    globals().pop(_obj, None)


In [ ]:
# /*SE INCORPORA EN CIERRE 2022Q2. DATA BONOS EMITIDOS EN EL EXTERIOR POR NO REGULADOS DEL
#   SECTOR FINANCIERO SECTOR=36912. EN CIERRE DE AÑO IMPUTAR TODA LA SERIE. CIERRE 2022: SE
#   INCORPORA AJUSTE PARA TODA LA SERIE PARA SER CONSISTENTES. CIERRE 2025Q2: INCORPORA EL EMISOR 33*/
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_rf_ext"))
sql_bonos_rf_ext = """
SELECT 'P' AS MONEDA, AÑO, TRIMESTRE AS TRIM, 36912 AS SECTOR,
       CAST('6' AS varchar(4)) AS C_CAGENTE, C_INSTRUMENTO_SCN AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       C_ENTRADA, C_CUENTA, SUM(DATO) * 1000 AS DATO, 'CI' AS FUENTE
INTO #bonos_rf_ext
FROM TABLAS.dbo.BASE_DEUDA_EMV
WHERE C_SI_EMISOR IN (36, 33) AND C_SI_TENEDOR = 6 AND VARIABLE NOT IN ('Valor Mercado MM$')
GROUP BY AÑO, TRIMESTRE, C_SI_EMISOR, C_INSTRUMENTO_SCN, C_ENTRADA, C_CUENTA
"""
work_conn.execute(text(sql_bonos_rf_ext))


In [ ]:
# UPDATE BONOS_RF_EXT SET C_CUENTA='Bce Final' WHERE C_CUENTA in ('Saldo Final');
# UPDATE BONOS_RF_EXT SET C_CUENTA='Financiera' WHERE C_CUENTA in ('Cuenta Financiera');
work_conn.execute(text("UPDATE #bonos_rf_ext SET C_CUENTA = 'Bce Final' WHERE C_CUENTA IN ('Saldo Final')"))
res = work_conn.execute(text("UPDATE #bonos_rf_ext SET C_CUENTA = 'Financiera' WHERE C_CUENTA IN ('Cuenta Financiera')"))
_log("UPDATE #bonos_rf_ext C_CUENTA", res.rowcount)


In [ ]:
# /**SELECCIONA BALANCE DE INICIO*/
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_rf_ext_bi"))
sql_bonos_bi = """
SELECT 'P' AS MONEDA,
       (CASE WHEN TRIMESTRE = 4 THEN AÑO + 1 ELSE AÑO END) AS AÑO,
       (CASE WHEN TRIMESTRE = 4 THEN 1 ELSE TRIMESTRE + 1 END) AS TRIM,
       36912 AS SECTOR, CAST('6' AS varchar(4)) AS C_CAGENTE,
       C_INSTRUMENTO_SCN AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       C_ENTRADA, 'Bce Inicio' AS C_CUENTA, SUM(DATO) * 1000 AS DATO, 'CI' AS FUENTE
INTO #bonos_rf_ext_bi
FROM TABLAS.dbo.BASE_DEUDA_EMV
WHERE C_SI_EMISOR IN (36, 33) AND C_SI_TENEDOR = 6
  AND VARIABLE NOT IN ('Valor Mercado MM$')
  AND C_CUENTA IN ('Saldo Final')
GROUP BY (CASE WHEN TRIMESTRE = 4 THEN AÑO + 1 ELSE AÑO END),
         (CASE WHEN TRIMESTRE = 4 THEN 1 ELSE TRIMESTRE + 1 END),
         C_SI_EMISOR, C_INSTRUMENTO_SCN, C_ENTRADA, C_CUENTA
"""
work_conn.execute(text(sql_bonos_bi))


In [ ]:
# data BONOS_RF_EXT; set BONOS_RF_EXT BONOS_RF_EXT_BI;
_cols_bonos = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_SCN, N_SCN, C_ENTRADA, C_CUENTA, DATO, FUENTE"
res = work_conn.execute(text(f"INSERT INTO #bonos_rf_ext ({_cols_bonos}) SELECT {_cols_bonos} FROM #bonos_rf_ext_bi"))
_log("INSERT #bonos_rf_ext <- #bonos_rf_ext_bi", res.rowcount)


In [ ]:
# /*CALCULA REC PRECIO*/
work_conn.execute(text("DROP TABLE IF EXISTS #rp_auxfin"))
sql_rp_auxfin = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_SCN, N_SCN, C_ENTRADA,
       'Rec Precio' AS C_CUENTA,
       SUM(CASE WHEN C_CUENTA NOT IN ('Bce Final') THEN DATO * -1 ELSE DATO END) AS DATO,
       FUENTE
INTO #rp_auxfin
FROM #bonos_rf_ext
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_SCN, N_SCN, C_ENTRADA, FUENTE
"""
work_conn.execute(text(sql_rp_auxfin))


In [ ]:
# data BONOS_RF_EXT; set BONOS_RF_EXT RP_AUXFIN;
res = work_conn.execute(text(f"INSERT INTO #bonos_rf_ext ({_cols_bonos}) SELECT {_cols_bonos} FROM #rp_auxfin"))
_log("INSERT #bonos_rf_ext <- #rp_auxfin", res.rowcount)


In [ ]:
# PROC SQL; DROP TABLE RP_AUXFIN, BONOS_RF_EXT_BI;
for _tmp in ["#rp_auxfin", "#bonos_rf_ext_bi"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_tmp}"))


In [ ]:
# DATA TABLAS.AJ_VARIOS; SET TABLAS.AJ_VARIOS BONOS_RF_EXT;
# Igual que el bloque anterior de AJ_VARIOS: el DATA step conserva lo vigente y suma
# el bloque nuevo -> INSERT server-side desde la #tmp
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.AJ_VARIOS ({_cols_bonos}) SELECT {_cols_bonos} FROM #bonos_rf_ext"))
_log("APPEND TABLAS.dbo.AJ_VARIOS (bonos_rf_ext)", res.rowcount)


In [ ]:
# Lectura de control del resultado del bloque de bonos (WORK.BONOS_RF_EXT para nodos siguientes)
bonos_rf_ext = pd.read_sql(text("SELECT * FROM #bonos_rf_ext"), work_conn)
_log("bonos_rf_ext", bonos_rf_ext)


In [ ]:
# PROC SQL; UPDATE BONOS_RF_EXT SET C_SCN='AF.42', N_SCN='Préstamos a largo plazo',
#            C_CAGENTE='321', dato=dato*-1
# BONOS_RF_EXT es un WORK materializado como #bonos_rf_ext en la sesión (tramo anterior)
sql_upd_bonos_rf_ext = """
UPDATE #bonos_rf_ext
SET C_SCN = 'AF.42',
    N_SCN = 'Préstamos a largo plazo',
    C_CAGENTE = '321',
    DATO = DATO * -1
"""
res = work_conn.execute(text(sql_upd_bonos_rf_ext))
_log("UPDATE #bonos_rf_ext", res.rowcount)


In [ ]:
# DATA TABLAS.AJ_VARIOS; SET TABLAS.AJ_VARIOS BONOS_RF_EXT; RUN;
# El SET concatena la tabla consigo misma más el WORK: en la base equivale a
# anexar las filas de #bonos_rf_ext a TABLAS.AJ_VARIOS (append server-side)
cols_aj_varios_bonos = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE"
sql_append_bonos = f"""
INSERT INTO TABLAS.dbo.AJ_VARIOS ({cols_aj_varios_bonos})
SELECT {cols_aj_varios_bonos}
FROM #bonos_rf_ext
"""
res = work_conn.execute(text(sql_append_bonos))
_log("APPEND TABLAS.dbo.AJ_VARIOS (BONOS_RF_EXT)", res.rowcount)


In [ ]:
# /*ELIMINA BONO DEL RM ACTIVO CON CA 36 DESDE 2022, PARA CONCILIAR BIEN CON LO IMPUTADO EN LA CI DE SECTOR 36*/
# CREATE TABLE AF32_51_6 AS /*LO QUE ESTA INICIALMENTE SE LLEVA A EMPRESAS*/
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51_6"))
sql_af32_51_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR, '51021' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN, T1.FUENTE
INTO #af32_51_6
FROM TABLAS.dbo.BD_CTSI_CIERRE T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6
       AND T1.C_CAGENTE = '36' AND T1.FUENTE = 'CI')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN,
         T1.SECTOR, T1.FUENTE
"""
res = work_conn.execute(text(sql_af32_51_6))
_log("SELECT INTO #af32_51_6", res.rowcount)


In [ ]:
# CREATE TABLE AF32_36_6 AS /*LO QUE ESTA INICIALMENTE SE RESTA, EXCEPTO BI DE TRIM=1*/
work_conn.execute(text("DROP TABLE IF EXISTS #af32_36_6"))
sql_af32_36_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR, T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN, T1.FUENTE
INTO #af32_36_6
FROM TABLAS.dbo.BD_CTSI_CIERRE T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6
       AND T1.C_CAGENTE = '36' AND T1.FUENTE = 'CI')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN,
         T1.SECTOR, T1.C_CAGENTE, T1.FUENTE
"""
res = work_conn.execute(text(sql_af32_36_6))
_log("SELECT INTO #af32_36_6", res.rowcount)


In [ ]:
# DATA TABLAS.AJ_VARIOS; SET TABLAS.AJ_VARIOS AF32_51_6 AF32_36_6; RUN;
# Anexa a la tabla las dos aperturas del ajuste AF.32 (a empresas y su contrapartida negativa)
cols_aj_varios_af32 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE"
for _tmp_af32 in ["#af32_51_6", "#af32_36_6"]:
    sql_append_af32 = f"""
    INSERT INTO TABLAS.dbo.AJ_VARIOS ({cols_aj_varios_af32})
    SELECT {cols_aj_varios_af32}
    FROM {_tmp_af32}
    """
    res = work_conn.execute(text(sql_append_af32))
    _log(f"APPEND TABLAS.dbo.AJ_VARIOS ({_tmp_af32})", res.rowcount)


In [ ]:
# PROC SQL; DROP TABLE AF32_51_6, AF32_36_6, AF32_36_6_VOL; QUIT;
# (AF32_36_6_VOL nunca se creó: su bloque está comentado en el SAS)
for _tmp_drop in ["#af32_51_6", "#af32_36_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_tmp_drop}"))


## S1_d

Transferencias de capital del gobierno general a empresas públicas, trimestralizadas en millones, reemplazando el periodo de coyuntura en la tabla principal

*confianza: medium · verificador: unverified · SAS: PROC SQL CREATE TABLE (WORK temporal + LEFT JOIN con agregación trimestral) + DELETE + PROC DATASETS APPEND*

In [ ]:
# ========= S1_d =========
# /*obtiene transferencias de capital a empresas para imputar en la síntesis*/
# /*año debe ser mayor o igual a 2005 en cierre de año y el corriente en coyuntura*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr"))
# ANIO interpolado como entero: un bind mataría la #tmp (sp_prepexec)
sql_ejec_cgr = f"""
SELECT *
INTO #ejec_cgr
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO >= {int(ANIO)}
"""
res = work_conn.execute(text(sql_ejec_cgr))
_log("#ejec_cgr", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #tk_ejec"))
sql_tk_ejec = """
SELECT t1.[AÑO],
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       5101 AS C_SI,
       'D.9' AS C_INSTRUMENTO_SCN,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(t1.DEVENG/1000000) AS Dato
INTO #tk_ejec
FROM #ejec_cgr t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
       ON  t1.C_PARTIDA     = t2.C_PARTIDA
       AND t1.C_CAPITULO    = t2.C_CAPITULO
       AND t1.C_PROGRAMA    = t2.C_PROGRAMA
       AND t1.C_ENTIDAD     = t2.ENTIDAD
       AND t1.C_TIPO_CUENTA = t2.T_CUENTA
       AND t1.C_CUENTA      = t2.C_CUENTA
       AND t1.C_ITEM        = t2.C_ITEM
       AND t1.C_ASIGNACION  = t2.C_ASIGNACION
       AND t1.C_ANALITICO   = t2.C_ANALITICO
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND (t2.OBS IS NULL OR t2.OBS NOT IN ('CR18_difcoy','CR18_difact'))
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('TC'))
  AND t1.MONEDA = 'P'
  AND t2.C_SCN IN ('D91','D92','D93','D99')
  AND t1.C_TIPO_CUENTA = 'G'
  AND t2.C_CAGENTE IN ('S11','S11_EPU','tkemppúb')
  AND t2.N_CUENTA = 'capital'
GROUP BY t1.[AÑO],
         CASE WHEN t1.MES IN (1,2,3) THEN 1
              WHEN t1.MES IN (4,5,6) THEN 2
              WHEN t1.MES IN (7,8,9) THEN 3
              ELSE 4 END
"""
res = work_conn.execute(text(sql_tk_ejec))
_log("#tk_ejec", res.rowcount)


In [ ]:
# /*ELIMINA DATOS DE AÑO DE COYUNTURA EN TABLA PRINCIPAL*/
with engine.begin() as conn:
    res = conn.execute(text(f"DELETE FROM TABLAS.dbo.T_TK_GG_EPU WHERE AÑO >= {int(ANIO)}"))
    _log("DELETE TABLAS.dbo.T_TK_GG_EPU", res.rowcount)


In [ ]:
res = work_conn.execute(text(f"DELETE FROM #tk_ejec WHERE AÑO < {int(ANIO)}"))
_log("DELETE #tk_ejec", res.rowcount)


In [ ]:
# /*ANEXA TK DE COYUNTURA A TABLA PRINCIPAL*/
# APPEND server-side con columnas explícitas (equivale al FORCE de PROC APPEND)
cols_tk = "[AÑO], TRIM, C_SI, C_INSTRUMENTO_SCN, C_CUENTA, C_ENTRADA, Dato"
sql_append_tk = f"""
INSERT INTO TABLAS.dbo.T_TK_GG_EPU ({cols_tk})
SELECT {cols_tk}
FROM #tk_ejec
"""
res = work_conn.execute(text(sql_append_tk))
_log("APPEND TABLAS.dbo.T_TK_GG_EPU", res.rowcount)


In [ ]:
# proc sql; delete EJEC_CGR,TK_EJEC; -> descarta los temporales de la sesión
for t in ["#ejec_cgr", "#tk_ejec"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## S1_b

Porcentaje de depósitos de hogares en fondos mutuos por año, trimestre y sector, completando 2003-2005 con el valor de 2006 y reemplazando desde el año de proceso en la base histórica

*confianza: medium · verificador: approve · SAS: PROC IMPORT XLSX + PROC SQL (CREATE TABLE, UPDATE, DELETE) + DATA step SET + PROC DATASETS APPEND*

In [ ]:
# ========= S1_b =========
# /*CALCULA % DE DEPÓSITOS PARA HOGARES EN FFMM*/
# IDENTIFICA_FFMM.sas7bdat ya es tabla de la base (TABLAS.IDENTIFICA_FFMM)
work_conn.execute(text("DROP TABLE IF EXISTS #id_fm"))
sql_id_fm = """
SELECT RUN_FONDO,
       CASE WHEN TIPO_FFMM IN (1, 2) THEN 3390101 ELSE 3390102 END AS SECTOR,
       COUNT(TIPO_FFMM) AS C
INTO #id_fm
FROM TABLAS.dbo.IDENTIFICA_FFMM
GROUP BY RUN_FONDO, TIPO_FFMM
"""
res = work_conn.execute(text(sql_id_fm))
_log("#id_fm", res.rowcount)


In [ ]:
# /*1. CALCULA % A HOGARES POR ROL DEL FONDO*/
ruta_bd_patrimonio = Path("data") / "mensual" / "BD_Patrimonio.xlsx"
base_datos = pd.read_excel(ruta_bd_patrimonio, sheet_name="base_datos")
_log("base_datos", base_datos)


In [ ]:
# base_datos vive en pandas (viene del Excel): se sube a la sesión como #tmp
# para que el SQL siguiente pueda operarla server-side
base_datos.to_sql("#base_datos", work_conn, if_exists="replace", index=False)
res = work_conn.execute(text("UPDATE #base_datos SET MES = 12 WHERE AÑO < 2017 AND MES IS NULL"))
_log("UPDATE #base_datos MES=12", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #dato_hh"))
sql_dato_hh = """
SELECT AÑO, MES, RUN,
       SUBSTRING(RUN, 1, 4) AS RUN_SDV,
       SUM(CASE WHEN DESTINO = 'EMPRE_HOGAR' THEN PATRI_T / 2 ELSE PATRI_T END) AS DATO,
       TIPO_FONDO
INTO #dato_hh
FROM #base_datos
WHERE DESTINO IN ('HOGARES', 'EMPRE_HOGAR')
GROUP BY AÑO, MES, RUN, TIPO_FONDO
"""
res = work_conn.execute(text(sql_dato_hh))
_log("#dato_hh", res.rowcount)


In [ ]:
# /*DATO TOTAL*/
work_conn.execute(text("DROP TABLE IF EXISTS #dato_tot"))
sql_dato_tot = """
SELECT AÑO, MES, RUN,
       SUBSTRING(RUN, 1, 4) AS RUN_SDV,
       SUM(PATRI_T) AS DATO
INTO #dato_tot
FROM #base_datos
GROUP BY AÑO, MES, RUN
"""
res = work_conn.execute(text(sql_dato_tot))
_log("#dato_tot", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #dato_est"))
sql_dato_est = """
SELECT T1.AÑO, T1.MES, T1.RUN,
       TRY_CAST(T1.RUN_SDV AS float) AS RUN_SDV,
       T1.TIPO_FONDO,
       T1.DATO / T2.DATO AS EST
INTO #dato_est
FROM #dato_hh T1, #dato_tot T2
WHERE T1.AÑO = T2.AÑO AND T1.MES = T2.MES AND T1.RUN = T2.RUN
"""
res = work_conn.execute(text(sql_dato_est))
_log("#dato_est", res.rowcount)


In [ ]:
# DROP TABLE base_datos, DATO_HH, DATO_TOT
for _t in ["#base_datos", "#dato_hh", "#dato_tot"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# /*2. OBTIENE DATOS A APLICAR PORCENTAJE*/
# CARTERA_INV_NACIONAL / CARTERA_INV_EXTERNA ya son tablas de la base
work_conn.execute(text("DROP TABLE IF EXISTS #data_fm_nac"))
work_conn.execute(text("SELECT * INTO #data_fm_nac FROM TABLAS.dbo.CARTERA_INV_NACIONAL"))
work_conn.execute(text("DROP TABLE IF EXISTS #data_fm_ext"))
res = work_conn.execute(text("SELECT * INTO #data_fm_ext FROM TABLAS.dbo.CARTERA_INV_EXTERNA"))
_log("#data_fm_ext", res.rowcount)


In [ ]:
# DATA data_fm; SET data_fm_NAC data_fm_ext (DROP=VALOR_REL_VAL);
# columnas comunes = las de #data_fm_nac menos VALOR_REL_VAL (que el SAS descarta
# del segundo dataset); se resuelven en runtime desde la metadata de la sesión
_cols_nac = pd.read_sql(text("SELECT TOP 0 * FROM #data_fm_nac"), work_conn).columns.tolist()
_cols_ext = pd.read_sql(text("SELECT TOP 0 * FROM #data_fm_ext"), work_conn).columns.tolist()
_cols_ext = [c for c in _cols_ext if c.upper() != "VALOR_REL_VAL"]
cols_data_fm = [c for c in _cols_nac if c in _cols_ext]
_sel_data_fm = ", ".join(cols_data_fm)
work_conn.execute(text("DROP TABLE IF EXISTS #data_fm"))
work_conn.execute(text(f"SELECT {_sel_data_fm} INTO #data_fm FROM #data_fm_nac"))
res = work_conn.execute(text(f"INSERT INTO #data_fm ({_sel_data_fm}) SELECT {_sel_data_fm} FROM #data_fm_ext"))
_log("#data_fm append externa", res.rowcount)


In [ ]:
for _t in ["#data_fm_nac", "#data_fm_ext"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #data_dep_run"))
sql_data_dep_run = """
SELECT T1.AÑO, T1.MES, T1.RUN_FONDO, T2.SECTOR,
       SUM(T1.VALOR_MERCADO) AS DATO
INTO #data_dep_run
FROM #data_fm T1
LEFT JOIN #id_fm T2 ON T1.RUN_FONDO = T2.RUN_FONDO
WHERE T1.T_INSTCORTO IN ('DPC', 'DPL') AND T1.MES IN (3, 6, 9, 12)
GROUP BY T1.AÑO, T1.MES, T1.RUN_FONDO, T2.SECTOR
"""
res = work_conn.execute(text(sql_data_dep_run))
_log("#data_dep_run", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #data_dep"))
sql_data_dep = """
SELECT AÑO, MES / 3 AS TRIM, SECTOR, SUM(DATO) AS DATO
INTO #data_dep
FROM #data_dep_run
GROUP BY AÑO, MES, SECTOR
"""
res = work_conn.execute(text(sql_data_dep))
_log("#data_dep", res.rowcount)


In [ ]:
# /*3. CALCULA DEPOSITOS A HOGARES*/
work_conn.execute(text("DROP TABLE IF EXISTS #dep_hh"))
sql_dep_hh = """
SELECT T1.AÑO, T1.MES / 3 AS TRIM, T2.SECTOR,
       SUM(T1.EST * T2.DATO) AS DATO
INTO #dep_hh
FROM #dato_est T1, #data_dep_run T2
WHERE T1.AÑO = T2.AÑO AND T1.MES = T2.MES AND T1.RUN_SDV = T2.RUN_FONDO
  AND T1.AÑO >= 2017
GROUP BY T1.AÑO, T1.MES, T2.SECTOR
"""
res = work_conn.execute(text(sql_dep_hh))
_log("#dep_hh", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #dep_hh_2"))
sql_dep_hh_2 = """
SELECT T2.AÑO, T2.MES / 3 AS TRIM, T2.SECTOR,
       SUM(T1.EST * T2.DATO) AS DATO
INTO #dep_hh_2
FROM #dato_est T1, #data_dep_run T2
WHERE T1.AÑO = T2.AÑO AND T1.RUN_SDV = T2.RUN_FONDO AND T2.AÑO < 2017
GROUP BY T2.AÑO, T2.MES, T2.SECTOR
"""
res = work_conn.execute(text(sql_dep_hh_2))
_log("#dep_hh_2", res.rowcount)


In [ ]:
# data DEP_HH; set DEP_HH DEP_HH_2; -> misma estructura (AÑO, TRIM, SECTOR, DATO)
cols_dep_hh = "AÑO, TRIM, SECTOR, DATO"
res = work_conn.execute(text(f"INSERT INTO #dep_hh ({cols_dep_hh}) SELECT {cols_dep_hh} FROM #dep_hh_2"))
_log("#dep_hh append #dep_hh_2", res.rowcount)


In [ ]:
# /*% total para dep de hogares*/
work_conn.execute(text("DROP TABLE IF EXISTS #dep_hh_fm"))
sql_dep_hh_fm = """
SELECT T1.AÑO, T1.TRIM, T1.SECTOR,
       T1.DATO / T2.DATO AS DATO
INTO #dep_hh_fm
FROM #dep_hh T1, #data_dep T2
WHERE T1.AÑO = T2.AÑO AND T1.TRIM = T2.TRIM AND T1.SECTOR = T2.SECTOR
"""
res = work_conn.execute(text(sql_dep_hh_fm))
_log("#dep_hh_fm", res.rowcount)


In [ ]:
# /*genera años faltantes*/
work_conn.execute(text("DROP TABLE IF EXISTS #imputa"))
work_conn.execute(text("""
SELECT CAST(2005 AS int) AS AÑO, T1.TRIM, T1.SECTOR, T1.DATO
INTO #imputa
FROM #dep_hh_fm T1
WHERE T1.AÑO = 2006 AND T1.TRIM = 1
"""))
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_2"))
work_conn.execute(text("""
SELECT CAST(2004 AS int) AS AÑO, T1.TRIM, T1.SECTOR, T1.DATO
INTO #imputa_2
FROM #dep_hh_fm T1
WHERE T1.AÑO = 2006 AND T1.TRIM = 1
"""))
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_3"))
res = work_conn.execute(text("""
SELECT CAST(2003 AS int) AS AÑO, T1.TRIM, T1.SECTOR, T1.DATO
INTO #imputa_3
FROM #dep_hh_fm T1
WHERE T1.AÑO = 2006 AND T1.TRIM = 1
"""))
_log("#imputa_3", res.rowcount)


In [ ]:
# data imputa; set imputa imputa_2 imputa_3; -> imputa queda con 2005 + 2004 + 2003
cols_imputa = "AÑO, TRIM, SECTOR, DATO"
for _t in ["#imputa_2", "#imputa_3"]:
    work_conn.execute(text(f"INSERT INTO #imputa ({cols_imputa}) SELECT {cols_imputa} FROM {_t}"))
_log("#imputa", pd.read_sql(text("SELECT * FROM #imputa"), work_conn))


In [ ]:
# imputa_a / imputa_b / imputa_c: copias de imputa con TRIM 2, 3 y 4
for _t, _trim in [("#imputa_a", 2), ("#imputa_b", 3), ("#imputa_c", 4)]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))
    work_conn.execute(text(f"SELECT {cols_imputa} INTO {_t} FROM #imputa"))
    work_conn.execute(text(f"UPDATE {_t} SET TRIM = {int(_trim)}"))


In [ ]:
# /*genera base serie completa*/
for _t in ["#imputa", "#imputa_a", "#imputa_b", "#imputa_c"]:
    work_conn.execute(text(f"INSERT INTO #dep_hh_fm ({cols_imputa}) SELECT {cols_imputa} FROM {_t}"))
_log("#dep_hh_fm serie completa", pd.read_sql(text("SELECT COUNT(*) AS N FROM #dep_hh_fm"), work_conn))


In [ ]:
# drop table id_fm, dato_est, data_fm, data_dep_run, data_dep, dep_hh, dep_hh_2,
# imputa, imputa_a, imputa_b, imputa_c, imputa_2, imputa_3
for _t in [
    "#id_fm", "#dato_est", "#data_fm", "#data_dep_run", "#data_dep",
    "#dep_hh", "#dep_hh_2", "#imputa", "#imputa_a", "#imputa_b", "#imputa_c",
    "#imputa_2", "#imputa_3",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# /*para recalcular años de coyuntura o serie completa si es cierre de año*/
with engine.begin() as conn:
    res = conn.execute(text(f"DELETE FROM TABLAS.dbo.DEP_HH_FM WHERE AÑO >= {int(anio)}"))
    _log("DELETE TABLAS.dbo.DEP_HH_FM", res.rowcount)


In [ ]:
# /*para recalcular años de coyuntura o serie completa si es cierre de año*/
res = work_conn.execute(text(f"DELETE FROM #dep_hh_fm WHERE AÑO < {int(anio)}"))
_log("DELETE #dep_hh_fm", res.rowcount)


In [ ]:
# proc datasets; append base=tablas.dep_hh_fm data=dep_hh_fm force;
# APPEND server-side: la fuente ya vive en la sesión, no baja a pandas
sql_append_dep_hh_fm = f"""
INSERT INTO TABLAS.dbo.DEP_HH_FM ({cols_imputa})
SELECT {cols_imputa}
FROM #dep_hh_fm
"""
res = work_conn.execute(text(sql_append_dep_hh_fm))
_log("APPEND TABLAS.dbo.DEP_HH_FM", res.rowcount)


In [ ]:
# proc sql; drop table dep_hh_fm;
work_conn.execute(text("DROP TABLE IF EXISTS #dep_hh_fm"))


## Bonos_Ext

Calcula los precios implícitos de los bonos externos por año, trimestre y sector (balance final y de inicio, con recompras y estimaciones) y reconstruye la base de precios de bonos externos

*confianza: medium · verificador: approve · SAS: PROC IMPORT XLSX (2 hojas) + PROC SQL CREATE/UPDATE con agregación y UNION ALL + DATA step de concatenación*

In [ ]:
# ========= Bonos_Ext =========
# /*DATA DE BONOS DE LA BALANZA DE PAGOS PARA CALCULAR PRECIOS IMPLÍCITOS A USAR EN LAS CUENTAS DE GOBIERNO, EMPRESAS Y HOLDINGS*/
# /*1. IMPORTA DATA*/
ruta_bonos_ext = Path("data") / "INFO_AUX" / "bonos_ext_cdr18.xlsx"
bonos_ext = pd.read_excel(ruta_bonos_ext, sheet_name="DATA")
_log("bonos_ext", bonos_ext)


In [ ]:
# UPDATE BONOS_EXT ... (recodificaciones previas al cálculo de precios)
bonos_ext.loc[bonos_ext["CNSI"] == 5102, "CNSI"] = 51021
bonos_ext.loc[bonos_ext["CNSI"] == 322, "CNSI"] = 321
bonos_ext.loc[bonos_ext["C_CAGENTE"].astype("string") != "6", "C_CAGENTE"] = "6"
# /*incorporado cierre 2025q2*/
bonos_ext.loc[bonos_ext["CNSI"] == 33, "CNSI"] = 36
_log("bonos_ext", bonos_ext)


In [ ]:
bonos_ext_est = pd.read_excel(ruta_bonos_ext, sheet_name="DATA_EST")
# delete from BONOS_EXT_EST where año=. -> el faltante de SAS es nulo
bonos_ext_est = bonos_ext_est[bonos_ext_est["Año"].notna()].copy()
_log("bonos_ext_est", bonos_ext_est)


In [ ]:
# Los WORK nacidos del Excel se materializan como #tmp para operar en la sesión SQL
bonos_ext.to_sql("#bonos_ext", work_conn, if_exists="replace", index=False)
bonos_ext_est.to_sql("#bonos_ext_est", work_conn, if_exists="replace", index=False)
_log("#bonos_ext / #bonos_ext_est", len(bonos_ext) + len(bonos_ext_est))


In [ ]:
# /*CALCULA PRECIOS PARA GOBIERNO, EMPRESAS Y HOLDINGS*/
# /*CIERRE 2021: INCORPORA TMB BANCOS*/
# /*CIERRE 2022Q2: INCORPORA PRECIO DE BNOS EMITIDOS EN EL EXTERIOR DE AUXILIARES (36)*/
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_precio_bf"))
sql_bonos_ext_precio_bf = """
SELECT [Año], [Trim], CNSI AS Sector, C_CAGENTE, C_SCN, C_CUENTA,
       SUM(Valor_Mercado) / SUM(Valor_par) AS Precio
INTO #bonos_ext_precio_bf
FROM #bonos_ext
WHERE fuente IN ('Mercado Externo', 'Mercado Externo (Recompras)')
  AND CNSI IN (41, 37, 5101, 51021, 5102, 321, 36)
GROUP BY [Año], [Trim], CNSI, C_CAGENTE, C_SCN, C_CUENTA
"""
res = work_conn.execute(text(sql_bonos_ext_precio_bf))
_log("#bonos_ext_precio_bf", res.rowcount)


In [ ]:
# create table Bonos_Ext_Recompra /*Balance Final, para recompras en empresas*/
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_recompra"))
sql_bonos_ext_recompra = """
SELECT [Año], [Trim], CNSI AS Sector, '54' AS C_CAGENTE, C_SCN, C_CUENTA,
       SUM(Valor_Mercado) / SUM(Valor_par) AS Precio
INTO #bonos_ext_recompra
FROM #bonos_ext
WHERE fuente IN ('Mercado Externo (Recompras)')
  AND CNSI IN (5101, 51021)
GROUP BY [Año], [Trim], CNSI, C_CAGENTE, C_SCN, C_CUENTA
"""
res = work_conn.execute(text(sql_bonos_ext_recompra))
_log("#bonos_ext_recompra", res.rowcount)


In [ ]:
# update Bonos_Ext_Recompra set Precio=1 where Precio=. -> el faltante de SAS es NULL en SQL Server
res = work_conn.execute(text("UPDATE #bonos_ext_recompra SET Precio = 1 WHERE Precio IS NULL"))
_log("UPDATE #bonos_ext_recompra Precio=1", res.rowcount)


In [ ]:
# data Bonos_Ext_Precio; set Bonos_Ext_Recompra Bonos_Ext_Precio BONOS_EXT_EST; -> concatenación en el mismo orden
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_precio"))
cols_precio = "[Año], [Trim], Sector, C_CAGENTE, C_SCN, C_CUENTA, Precio"
sql_bonos_ext_precio = f"""
SELECT {cols_precio} INTO #bonos_ext_precio FROM #bonos_ext_recompra
UNION ALL
SELECT {cols_precio} FROM #bonos_ext_precio_bf
UNION ALL
SELECT {cols_precio} FROM #bonos_ext_est
"""
res = work_conn.execute(text(sql_bonos_ext_precio))
_log("#bonos_ext_precio", res.rowcount)


In [ ]:
# create table Bonos_Ext_Precio_2 /*Balance Inicio*/ -> desfase trimestral (TRIM+1)
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_precio_2"))
sql_bonos_ext_precio_2 = """
SELECT (CASE WHEN [Trim] = 4 THEN [Año] + 1 ELSE [Año] END) AS [Año],
       (CASE WHEN [Trim] = 4 THEN 1 ELSE [Trim] + 1 END) AS [Trim],
       Sector, C_CAGENTE, C_SCN, 'Bce Inicio' AS C_CUENTA, Precio
INTO #bonos_ext_precio_2
FROM #bonos_ext_precio
"""
res = work_conn.execute(text(sql_bonos_ext_precio_2))
_log("#bonos_ext_precio_2", res.rowcount)


In [ ]:
# DATA tablas.Bonos_Ext_Precio; SET Bonos_Ext_Precio_2 Bonos_Ext_Precio;
# La tabla la crea este flujo en cada corrida (created_tables): se arma de cero
work_conn.execute(text("DROP TABLE IF EXISTS TABLAS.dbo.BONOS_EXT_PRECIO"))
sql_crea_bonos_ext_precio = f"""
SELECT {cols_precio} INTO TABLAS.dbo.BONOS_EXT_PRECIO FROM #bonos_ext_precio_2
UNION ALL
SELECT {cols_precio} FROM #bonos_ext_precio
"""
res = work_conn.execute(text(sql_crea_bonos_ext_precio))
_log("CREATE TABLAS.dbo.BONOS_EXT_PRECIO", res.rowcount)


In [ ]:
# update tablas.Bonos_Ext_Precio set Sector=36912 where Sector=36 -> contra la tabla ya creada
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BONOS_EXT_PRECIO SET Sector = 36912 WHERE Sector = 36"))
    _log("UPDATE TABLAS.dbo.BONOS_EXT_PRECIO Sector=36912", res.rowcount)


In [ ]:
# drop table Bonos_Ext_Precio, BONOS_EXT_EST, Bonos_Ext_Precio_2, bonos_ext, Bonos_Ext_Recompra
for _t in ["#bonos_ext_precio", "#bonos_ext_est", "#bonos_ext_precio_2", "#bonos_ext", "#bonos_ext_recompra", "#bonos_ext_precio_bf"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))
